In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [5]:

from google.colab import files
uploaded = files.upload()

Saving X_rank3_tensor.npy to X_rank3_tensor.npy
Saving y_labels.npy to y_labels.npy


In [6]:
import numpy as np

x_raw = np.load('X_rank3_tensor.npy')
y_raw = np.load('y_labels.npy')

x_raw.shape

(1765, 60, 10)

In [7]:
print(y_raw[:10], y_raw.shape)

['ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP'] (1765,)


In [9]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

track_features = [
		'tempo', 'loudness', 'mode', 'danceability', 'energy',
    'speechiness', 'acousticness', 'liveness', 'happiness', 'instrumentalness'
]

numeric_features = [f for f in track_features if f != 'mode']
categorical_feature = ['mode']

scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)


def data_preprocessed(x_data, is_train=False):
	(n_playlists, n_tracks, n_features) = x_data.shape
	x_2d = x_data.reshape(n_playlists*n_tracks, n_features)
	x_df = pd.DataFrame(x_2d, columns=track_features)

	real_song = x_df['tempo'] > 0

	if is_train:
		scaler.fit(x_df.loc[real_song, numeric_features])

	x_df.loc[real_song, numeric_features] = scaler.transform(
		x_df.loc[real_song, numeric_features]
	)

	if is_train:
		encoder.fit(x_df.loc[real_song, categorical_feature])

	mode_encoded = encoder.transform(x_df[categorical_feature])
	mode_encoded[~real_song] = 0

	x_numeric = x_df[numeric_features].values
	x_2d = np.concatenate([x_numeric, mode_encoded], axis=1)

	n_features = x_2d.shape[1]
	x_data = x_2d.reshape(n_playlists, n_tracks, n_features)
	return x_data

x_data = data_preprocessed(x_raw, is_train=True)
print("(전처리완료) x_data shape: ", x_data.shape)



(전처리완료) x_data shape:  (1765, 60, 11)


In [10]:
original_features = [
    'tempo', 'loudness', 'mode', 'danceability', 'energy',
    'speechiness', 'acousticness', 'liveness', 'happiness', 'instrumentalness'
]

selected_features = ['tempo', 'danceability', 'loudness', 'energy', 'acousticness']
selected_idx = [original_features.index(f) for f in selected_features]

x_raw_small = x_raw[:, :, selected_idx]
print("shape:", x_raw_small.shape)

track_features = selected_features
numeric_features = selected_features
categorical_feature = []


def data_preprocessed(x_data, is_train=False):
    (n_playlists, n_tracks, n_features) = x_data.shape

    x_2d = x_data.reshape(n_playlists*n_tracks, n_features)
    x_df = pd.DataFrame(x_2d, columns=track_features)

    real_song = x_df['tempo'] > 0

    if is_train:
        scaler.fit(x_df.loc[real_song, numeric_features])

    x_df.loc[real_song, numeric_features] = scaler.transform(
        x_df.loc[real_song, numeric_features]
    )

    x_numeric = x_df[numeric_features].values
    x_2d = x_numeric

    x_data = x_2d.reshape(n_playlists, n_tracks, len(numeric_features))
    return x_data


x_data = data_preprocessed(x_raw_small, is_train=True)
print("전처리 완료 shape:", x_data.shape)

shape: (1765, 60, 5)
전처리 완료 shape: (1765, 60, 5)


## 2) 하나의 CNN에서 4개의 출력을 동시에 내는 Multitask

In [79]:
import numpy as np

def mbti_to_multitask_label(y_mbti):
    labels = []
    for mbti in y_mbti:
        ei = 1 if mbti[0] == 'E' else 0
        sn = 1 if mbti[1] == 'N' else 0
        ft = 1 if mbti[2] == 'T' else 0
        pj = 1 if mbti[3] == 'J' else 0
        labels.append([ei, sn, ft, pj])
    return np.array(labels)

y_multi = mbti_to_multitask_label(y_raw)

y_multi.shape

(1765, 4)

In [80]:

idx_trainval, idx_test = train_test_split(
    np.arange(len(x_data)),
    test_size=0.2,
    random_state=42,
    stratify=y_multi[:, 0]
)


idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_multi[idx_trainval, 0]
)


x_train = x_data[idx_train]
x_val   = x_data[idx_val]
x_test  = x_data[idx_test]

y_train = y_multi[idx_train]
y_val   = y_multi[idx_val]
y_test  = y_multi[idx_test]

In [244]:
import numpy as np
import tensorflow.keras.backend as K
import tensorflow as tf
from tensorflow.keras import layers, models

# 축별 가중치 설정하여 학습
raw_weights = np.array([1.0, 1.0, 1.0, 1.2], dtype=np.float32)
LOSS_WEIGHTS = K.constant(raw_weights / raw_weights.mean())

def weighted_binary_crossentropy(y_true, y_pred):

    bce = K.binary_crossentropy(y_true, y_pred)
    weighted_bce = bce * LOSS_WEIGHTS
    return K.mean(K.sum(weighted_bce, axis=-1))

In [243]:
from tensorflow.keras.regularizers import l2

def build_multitask_cnn_l2():
  model = Sequential([
      Conv1D(32, kernel_size= 3, activation='relu', padding='same',
             kernel_regularizer=l2(1e-4),
             input_shape=(60,5)),
      MaxPooling1D(pool_size=2),

      Conv1D(64, kernel_size=3, activation='relu', padding='same',
             kernel_regularizer=l2(1e-4)),
      MaxPooling1D(pool_size=2),

      Flatten(),

      Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
      Dropout(0.3),

      Dense(4, activation='sigmoid')

  ])

  model.compile(optimizer='adam',
                loss=weighted_binary_crossentropy,
                metrics=['binary_accuracy'])
  return model

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

In [ ]:
model_mt = build_multitask_cnn_l2()

history_mt = model_mt.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_mt.history['loss'], label='Train Loss')
plt.plot(history_mt.history['val_loss'], label='Val Loss')
plt.title("Multitask Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()


plt.subplot(1, 2, 2)
plt.plot(history_mt.history['binary_accuracy'], label='Train Accuracy')
plt.plot(history_mt.history['val_binary_accuracy'], label='Val Accuracy')
plt.title("Multitask Model Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 테스트 데이터

model_mt.evaluate(x_test, y_test)

pred = (model_mt.predict(x_test) > 0.5).astype(int)

ei_acc = (pred[:,0] == y_test[:,0]).mean()
sn_acc = (pred[:,1] == y_test[:,1]).mean()
ft_acc = (pred[:,2] == y_test[:,2]).mean()
pj_acc = (pred[:,3] == y_test[:,3]).mean()

def combine_mbti(ei, sn, ft, pj):
    return (
        ("E" if ei==1 else "I") +
        ("N" if sn==1 else "S") +
        ("T" if ft==1 else "F") +
        ("J" if pj==1 else "P")
    )

pred_mbti = [combine_mbti(*row) for row in pred]

full_acc = (np.array(pred_mbti) == y_raw[idx_test]).mean()

# 4)
def partial_accuracy(pred_list, true_list):
    total = 0
    for p, t in zip(pred_list, true_list):
        total += sum([p[i] == t[i] for i in range(4)]) / 4
    return total / len(pred_list)